In [ ]:
# Landmark Recognition using Simple CNN
# - Dataset: Google Landmark Recognition 2020 (Top 10 Classes Sampled)
# - Model: Custom SimpleCNN
# - Acccuracy: 72.00%

# <<<< 데이터 포장 >>>>

# 1. 시스템 및 파일 관리
import os
import glob
import random

# 2. 데이터 분석 및 행렬 연산
import numpy as np
import pandas as pd

# 3. 시각화 및 이미지 처리
import cv2
import matplotlib.pyplot as plt
from PIL import Image

# 4. 머신러닝/딥러닝 프레임워크 (PyTorch)
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

# 5. 성능 평가 및 데이터 분리
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

csv_path = '/kaggle/input/landmark-recognition-2020-128x128/landmark_recognition_128/train.csv'
train_df = pd.read_csv(csv_path)

# 6. 환경 확인
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"현재 사용 중인 장치: {device}") # GPUT4 x2 사용

# [유명한 Top 10 랜드마크만 골라내어 데이터 포장]
# 1. 데이터가 가장 많은 상위 10개 랜드마크 ID 추출
top_n = 10 
top_landmark_ids = train_df['landmark_id'].value_counts().nlargest(top_n).index

# 2. 해당 랜드마크 데이터만 필터링 (같은 랜드마크 사진이 여러 장으로 묶음)
train_sample = train_df[train_df['landmark_id'].isin(top_landmark_ids)].copy().reset_index(drop=True)

# (옵션) 학습 속도를 위해 데이터를 2000초과로 받더라도 2000개 고정으로 잘라냄
if len(train_sample) > 2000: 
    train_sample = train_sample.sample(n=2000, random_state=42).reset_index(drop=True)

print(f"선택된 데이터 개수: {len(train_sample)}개")

# 3. 라벨 압축 - 0~9번으로 다시 번호 매기기
unique_labels = train_sample['landmark_id'].unique()
label_map = {raw_id: idx for idx, raw_id in enumerate(unique_labels)}
train_sample['label_idx'] = train_sample['landmark_id'].map(label_map) 

# 4. 클래스 개수 저장
num_classes = len(unique_labels)
print(f"총 클래스 개수(num_classes): {num_classes}")

# 5. 데이터 분리 
# stratify=... : 훈련/검증 셋에 정답 비율을 똑같이 맞춰줌
train_data, val_data = train_test_split(train_sample, 
                                        test_size=0.2, 
                                        random_state=42, 
                                        stratify=train_sample['label_idx'])

# ★★★ __init__, __len__, __getitem__ => 파이토치의 커스텀 데이터셋 기본 템플릿 
class LandmarkDataset(Dataset) :
    # init은 getitem을 위한 사전 준비, len은 getitem을 위한 범위 제한
    def __init__(self, df, transform=None):
        self.df = df
        # 사진들이 들어있는 시작 폴더 주소를 잡기 위해 폴더까지만 지정 (csv_path는 포함 X)
        self.base_dir = '/kaggle/input/landmark-recognition-2020-128x128/landmark_recognition_128/train'
        self.transform = transform

    def __len__(self) :
        return len(self.df)

    def __getitem__(self, idx) : #실질적 실행자
        # 1. CSV(리스트)에서 정보 꺼내기
        row = self.df.iloc[idx]
        img_id = row['id']
        label = row['label_idx'] # 아까 만든 압축 라벨 사용
        
        # 2. 창고(base_dir)에서 실제 이미지 경로 만들기
        # (이미지가 a/b/c/abc.jpg 로 구성되어 맞게 조정)
        img_path = os.path.join(self.base_dir, img_id[0], img_id[1], img_id[2], f"{img_id}.jpg")

        image = Image.open(img_path).convert('RGB') #이미지 열어
        if self.transform:
            image = self.transform(image) # 아래에 정의한 크기조절, 숫자 행렬로 변환, 정규화 과정 실행
            
        return image, torch.tensor(label, dtype=torch.long)

transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# 학습용, 시험용(학습용 보다 같거나 크게 잡음 ex: 64, 128)
# batch_size : 모델이 한번에 공부하는 문제의 양
train_loader = DataLoader(LandmarkDataset(train_data, transform), batch_size=32, shuffle=True) 
# train 중에는 문제 순서 외우지 못하게 shuffle로 섞음
val_loader = DataLoader(LandmarkDataset(val_data, transform), batch_size = 32, shuffle=False)
# 시험 중에는 굳이 섞을 필요 없이 정답지와 대조만 하면됨 
# 속도 높이려면 num_workers=2 / pin_memory=True(메모리->gpu로 고속이동) 

현재 사용 중인 장치: cuda
선택된 데이터 개수: 2000개
총 클래스 개수(num_classes): 10


In [8]:
# <<<< 모델 설계 >>>>

class SimpleCNN(nn.Module) :
    def __init__(self, num_classes) :
        super(SimpleCNN, self).__init__()

        # conv:특징 추출기, Pool:데이터 압축기(큰값 or 평균 값 남기고 나머지 버림), FC: 최종 판단기 
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding = 1)
        # < 필터의 '틀'을 잡음 >
        # in_channels : 입력데이터의 깊이 (입력 채널 수)
        # out_channels : 서로 다른 필터의 개수 (출력 채널 수)
        # kernel_size : 필터의 크기
        # padding : 이미지 테두리에 0을 한 줄 두름 => 이미지 크기가 줄어드는 성질 
        # (연산 전후 이미지 해상도도 그대로 유지)  
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding = 1)
        self.conv3 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding = 1)

        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        self.fc1 = nn.Linear(128*16*16, 512) # 가중치 점수 합산 시스템
        self.fc2 = nn.Linear(512, num_classes)

    def forward(self, x) : # self는 SimpleCNN 클래스 그 자체를 의미, x는 현재 공정중인 데이터를 의미
        # relu는 입력이 0보다 크면 통과, 0보다 작으면 0으로
        # relu는 신호의 강도 기준으로 필터링, pooling는 공간의 위치(정보 요약) 기준으로 압축
        x = self.pool(F.relu(self.conv1(x))) 
        # Relu로 양수들만 뽑아내고 pooling으로 MAX 뽑아냄
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))

        x = x.view(-1, 128 *16 * 16)

        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [ ]:
# <<<< 학습 루프 >>>>
model = SimpleCNN(num_classes=num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
 # model.parameters()에 RGB와 좌표의 조합. 즉, self.conv1에 설정한 필터의 수치를 포함함

num_epochs = 10

print(f"학습 시작! (총 클래스 개수: {num_classes})")

for epoch in range(num_epochs) :
    model.train() # 학습모드 on
    running_loss = 0.0

    # train
    for inputs, labels in train_loader : # 데이터 포장단계에서 선언함
        inputs, labels = inputs.to(device), labels.to(device) 

        optimizer.zero_grad() # 1. 기울기 초기화
        outputs = model(inputs) # 2. 예측
        loss = criterion(outputs, labels) # 3. 오차 계산
        loss.backward() # 4. 역전파 (어떤 필터 숫자가 문제였는지 계산)
        optimizer.step() # 5. 가중치 갱신 (필터 안의 숫자를 약간 수정함)

        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/{num_epochs}] Loss: {running_loss/len(train_loader):.4f}")

학습 시작! (총 클래스 개수: 10)
Epoch [1/10] Loss: 1.4209
Epoch [2/10] Loss: 0.8621
Epoch [3/10] Loss: 0.5558
Epoch [4/10] Loss: 0.3168
Epoch [5/10] Loss: 0.1886
Epoch [6/10] Loss: 0.0731
Epoch [7/10] Loss: 0.0493
Epoch [8/10] Loss: 0.0620
Epoch [9/10] Loss: 0.0366
Epoch [10/10] Loss: 0.0051


In [ ]:
# <<<< 최종 평가 (Evaluation) >>>> 

print("\n[최종 평가] 학습된 모델로 최종 성능을 측정합니다...")

# 1. 평가 모드 전환 (필수!)
model.eval() 

correct = 0
total = 0
all_labels = []
all_preds = []

# 2. 평가 진행 (기울기 계산 X)
with torch.no_grad():
    for inputs, labels in val_loader: # 데이터 포장 단계에서 선언함
        inputs, labels = inputs.to(device), labels.to(device)
        
        # 모델 예측
        outputs = model(inputs)
        
        # 가장 높은 확률의 정답 선택
        _, predicted = torch.max(outputs.data, 1)
        
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
        # 나중에 혼동 행렬(Confusion Matrix) 등을 그리려면 저장해둠
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(predicted.cpu().numpy())

# 3. 최종 결과 출력
final_acc = 100 * correct / total
print(f"===========================================")
print(f"🏆 최종 모델 정확도: {final_acc:.2f}%")
print(f"===========================================")


[최종 평가] 학습된 모델로 최종 성능을 측정합니다...
🏆 최종 모델 정확도: 72.00%
